# Get results out of my WandB automatically

In [4]:
import wandb
import numpy as np
from itertools import product
import pandas as pd
import re

api = wandb.Api()
runs = api.runs("labeebah-islaam/world_models")

# Procgen

In [6]:
# ----------------------------
# Config
# ----------------------------
ENTITY = "labeebah-islaam"
PROJECT = "world_models"

TARGET_STEPS = {5, 10}
TARGET_PPS = {0.05, 0.1, 0.2, 0.5}
TARGET_MODES = {"value", "reward"}

EP_LEN_KEY = "actor_critic/eval/episode_length"
PLANNED_REWARD_KEY = "actor_critic/eval/planned_cumulative_reward"
BASELINE_REWARD_KEY = "actor_critic/eval/cumulative_reward"

OUT_CSV = "procgen_planning_summary.csv"

# Example:
# StarPilot_5_roll_0_inner_0.1_pct_1_max_reward_seed_2_time_2026-01-27_04-06-38
NAME_RE = re.compile(
    r"^(?P<env>.+?)_"
    r"(?P<steps>\d+)_roll_"
    r"(?P<inner>\d+)_inner_"
    r"(?P<pp>[0-9]*\.?[0-9]+)_pct_"
    r"(?P<depth>\d+)_max_"
    r"(?P<mode>[^_]+)_seed_"
    r"(?P<seed>\d+)_time_"
)

# ----------------------------
# Helpers
# ----------------------------
def parse_run_name(name: str):
    if not name:
        return None
    m = NAME_RE.match(name)
    if not m:
        return None
    d = m.groupdict()
    return {
        "env_type": d["env"],
        "planning_steps": int(d["steps"]),
        "inner_steps": int(d["inner"]),
        "planning_percentage": float(d["pp"]),
        "planning_depth": int(d["depth"]),
        "planning_mode": d["mode"],
        "seed": int(d["seed"]),
    }

def float_close(a, b, tol=1e-8):
    return abs(float(a) - float(b)) <= tol

def get_final_logged_value(run, key):
    """
    Returns the last non-NaN logged value for a metric.
    Uses history() for simplicity; if you have very long runs and sampling
    becomes an issue, switch this to scan_history().
    """
    try:
        hist = run.history(keys=[key], pandas=True)
        if hist is None or hist.empty or key not in hist.columns:
            return None
        vals = hist[key].dropna().values
        if len(vals) == 0:
            return None
        return float(vals[-1])
    except Exception:
        return None

def mean_std(arr):
    if len(arr) == 0:
        return None, None
    arr = np.array(arr, dtype=float)
    return float(arr.mean()), float(arr.std(ddof=0))

# ----------------------------
# Load runs
# ----------------------------
api = wandb.Api()
runs = list(api.runs(f"{ENTITY}/{PROJECT}"))

print(f"Loaded {len(runs)} runs")

# ----------------------------
# Filter planning runs only
# ----------------------------
matched = []
bad_name = 0

for r in runs:
    if r.state != "finished":
        continue

    info = parse_run_name(r.name or "")
    if info is None:
        bad_name += 1
        continue

    if info["planning_steps"] not in TARGET_STEPS:
        continue
    if info["planning_mode"] not in TARGET_MODES:
        continue
    if not any(float_close(info["planning_percentage"], x) for x in TARGET_PPS):
        continue

    if info["planning_depth"] != 1: # No inner planning in Procgen for now
        continue

    matched.append((r, info))

print(f"Matched planning runs: {len(matched)}")
print(f"Name parse failed on finished runs: {bad_name}")

# ----------------------------
# Group by setting
# ----------------------------
# key = (env, planning_steps, planning_percentage, planning_mode)
groups = {}

for r, info in matched:
    key = (
        info["env_type"],
        info["planning_steps"],
        info["planning_percentage"],
        info["planning_mode"],
    )
    groups.setdefault(key, {})
    # keep first run per seed
    if info["seed"] not in groups[key]:
        groups[key][info["seed"]] = r

# ----------------------------
# Aggregate
# ----------------------------
rows = []

for (env_type, planning_steps, planning_percentage, planning_mode), by_seed in sorted(groups.items()):
    rs = list(by_seed.values())

    rewards_all = []
    ep_lens_all = []
    reward_per_step = []
    used_seeds_all = []
    used_seeds_ratio = []

    for r in rs:
        if r.state != "finished":
            continue

        reward_key = PLANNED_REWARD_KEY if planning_steps != 0 else BASELINE_REWARD_KEY

        final_reward = get_final_logged_value(r, reward_key)
        final_ep_len = get_final_logged_value(r, EP_LEN_KEY)

        if final_reward is None or final_ep_len is None:
            continue

        seed = parse_run_name(r.name)["seed"]

        rewards_all.append(final_reward)
        ep_lens_all.append(final_ep_len)
        used_seeds_all.append(seed)

        if final_ep_len != 0:
            ratio = final_reward / final_ep_len
            if not np.isnan(ratio) and not np.isinf(ratio):
                reward_per_step.append(ratio)
                used_seeds_ratio.append(seed)

    reward_mean, reward_std = mean_std(rewards_all)
    ep_mean, ep_std = mean_std(ep_lens_all)
    ratio_mean, ratio_std = mean_std(reward_per_step)

    row = {
        "env_type": env_type,
        "planning_steps": planning_steps,
        "planning_percentage": planning_percentage,
        "planning_mode": planning_mode,

        "n_seeds_all": len(rewards_all),
        "seeds_all": ",".join(map(str, sorted(used_seeds_all))),
        "reward_mean_all": reward_mean,
        "reward_std_all": reward_std,
        "episode_length_mean_all": ep_mean,
        "episode_length_std_all": ep_std,

        "n_seeds_ratio": len(reward_per_step),
        "seeds_ratio": ",".join(map(str, sorted(used_seeds_ratio))),
        "reward_per_step_mean": ratio_mean,
        "reward_per_step_std": ratio_std,
    }

    rows.append(row)

df = pd.DataFrame(rows).sort_values(
    ["env_type", "planning_steps", "planning_percentage", "planning_mode"]
).reset_index(drop=True)

print(df)
df.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}")

Loaded 4515 runs
Matched planning runs: 2294
Name parse failed on finished runs: 1817
      env_type  planning_steps  planning_percentage planning_mode  \
0      BigFish               5                 0.05        reward   
1      BigFish               5                 0.05         value   
2      BigFish               5                 0.10        reward   
3      BigFish               5                 0.10         value   
4      BigFish               5                 0.20        reward   
..         ...             ...                  ...           ...   
251  StarPilot              10                 0.10         value   
252  StarPilot              10                 0.20        reward   
253  StarPilot              10                 0.20         value   
254  StarPilot              10                 0.50        reward   
255  StarPilot              10                 0.50         value   

     n_seeds_all            seeds_all  reward_mean_all  reward_std_all  \
0          

In [3]:
# ----------------------------
# Config
# ----------------------------
ENTITY = "labeebah-islaam"
PROJECT = "world_models"

EP_LEN_KEY = "actor_critic/eval/episode_length"
BASELINE_REWARD_KEY = "actor_critic/eval/cumulative_reward"

OUT_CSV = "procgen_baseline_summary.csv"

# Baseline name pattern:
# ${env.env_type}_${evaluation.planning_steps}_roll_${evaluation.inner_planning_steps}_inner_${evaluation.planning_percentage}_pct_${evaluation.planning_depth}_max_${evaluation.planning_mode}_seed_${common.seed}_time_...
NAME_RE = re.compile(
    r"^(?P<env>.+?)_"
    r"(?P<steps>\d+)_roll_"
    r"(?P<inner>\d+)_inner_"
    r"(?P<pp>[0-9]*\.?[0-9]+)_pct_"
    r"(?P<depth>\d+)_max_"
    r"(?P<mode>[^_]+)_seed_"
    r"(?P<seed>\d+)_time_"
)

def parse_run_name(name: str):
    if not name:
        return None
    m = NAME_RE.match(name)
    if not m:
        return None
    d = m.groupdict()
    return {
        "env_type": d["env"],
        "planning_steps": int(d["steps"]),
        "inner_steps": int(d["inner"]),
        "planning_percentage": float(d["pp"]),
        "planning_depth": int(d["depth"]),
        "planning_mode": d["mode"],
        "seed": int(d["seed"]),
    }

def get_final_logged_value(run, key):
    try:
        hist = run.history(keys=[key], pandas=True)
        if hist is None or hist.empty or key not in hist.columns:
            return None
        vals = hist[key].dropna().values
        if len(vals) == 0:
            return None
        return float(vals[-1])
    except Exception:
        return None

def mean_std(arr):
    if len(arr) == 0:
        return None, None
    arr = np.array(arr, dtype=float)
    return float(arr.mean()), float(arr.std(ddof=0))

# ----------------------------
# Load runs
# ----------------------------
api = wandb.Api()
runs = list(api.runs(f"{ENTITY}/{PROJECT}"))

print(f"Loaded {len(runs)} runs")

# ----------------------------
# Filter baseline runs only
# ----------------------------
matched = []
bad_name = 0

for r in runs:
    if r.state != "finished":
        continue

    info = parse_run_name(r.name or "")
    if info is None:
        bad_name += 1
        continue

    if info["planning_steps"] != 0:
        continue

    matched.append((r, info))

print(f"Matched baseline runs: {len(matched)}")
print(f"Name parse failed on finished runs: {bad_name}")

# ----------------------------
# Group by env only
# ----------------------------
groups = {}  # env_type -> {seed: run}

for r, info in matched:
    env_type = info["env_type"]
    seed = info["seed"]

    groups.setdefault(env_type, {})
    if seed not in groups[env_type]:
        groups[env_type][seed] = r

# ----------------------------
# Aggregate
# ----------------------------
rows = []

for env_type, by_seed in sorted(groups.items()):
    rs = list(by_seed.values())

    rewards_all = []
    ep_lens_all = []
    reward_per_step = []
    used_seeds_all = []
    used_seeds_ratio = []

    for r in rs:
        final_reward = get_final_logged_value(r, BASELINE_REWARD_KEY)
        final_ep_len = get_final_logged_value(r, EP_LEN_KEY)

        if final_reward is None or final_ep_len is None:
            continue

        seed = parse_run_name(r.name)["seed"]

        rewards_all.append(final_reward)
        ep_lens_all.append(final_ep_len)
        used_seeds_all.append(seed)

        if final_ep_len != 0:
            ratio = final_reward / final_ep_len
            if not np.isnan(ratio) and not np.isinf(ratio):
                reward_per_step.append(ratio)
                used_seeds_ratio.append(seed)

    reward_mean, reward_std = mean_std(rewards_all)
    ep_mean, ep_std = mean_std(ep_lens_all)
    ratio_mean, ratio_std = mean_std(reward_per_step)

    row = {
        "env_type": env_type,

        "n_seeds_all": len(rewards_all),
        "seeds_all": ",".join(map(str, sorted(used_seeds_all))),
        "reward_mean_all": reward_mean,
        "reward_std_all": reward_std,
        "episode_length_mean_all": ep_mean,
        "episode_length_std_all": ep_std,

        "n_seeds_ratio": len(reward_per_step),
        "seeds_ratio": ",".join(map(str, sorted(used_seeds_ratio))),
        "reward_per_step_mean": ratio_mean,
        "reward_per_step_std": ratio_std,
    }

    rows.append(row)

df = pd.DataFrame(rows).sort_values(["env_type"]).reset_index(drop=True)

print(df)
df.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}")

NameError: name 're' is not defined

# Atari

In [1]:
modes = ["reward", "value"]
steps = [0, 1, 2, 5, 10, 15, 20]
thresholds = [2, 1.5, 1]
inner_steps = [0, 1, 2, 5]
metric = "actor_critic/eval/planned_cumulative_reward"

def get_last_value(run):
    try:
        hist = run.history(keys=[metric], pandas=True)
        if not hist.empty:
            return hist[metric].dropna().values[-1]
    except Exception:
        pass
    return None

# Filter runs that do NOT start with 'pruned'
runs = [r for r in runs if not r.name.startswith("pruned")]

results = []
for mode, step, thres, inner in product(modes, steps, thresholds, inner_steps):
    matched = []
    for run in runs:
        cfg = run.config
        if run.state != "finished":
            continue
        try:
            if (
                str(cfg["evaluation"]["planning_mode"]) == str(mode) and
                int(cfg["evaluation"]["planning_steps"]) == int(step) and
                float(cfg["evaluation"]["entropy_threshold"]) == float(thres) and
                int(cfg["evaluation"]["inner_planning_steps"]) == int(inner)
            ):
                val = get_last_value(run)
                if val is not None:
                    matched.append(val)
        except KeyError:
            continue

    if len(matched) >= 3:
        print(f"Matched {len(matched)} runs for mode={mode}, steps={step}, thres={thres}, inner={inner}")
        arr = np.array(matched[:3])  # only take first 3
        result = {
            "mode": mode,
            "planning_steps": step,
            "threshold": thres,
            "inner_steps": inner,
            "mean": arr.mean(),
            "std": arr.std()
        }
    else:
        result = {
            "mode": mode,
            "planning_steps": step,
            "threshold": thres,
            "inner_steps": inner,
            "mean": None,
            "std": None
        }

    results.append(result)

# Export
df = pd.DataFrame(results)
print(df)
df.to_csv("sweep_results.csv", index=False)

NameError: name 'runs' is not defined

In [ ]:
import numpy as np
import pandas as pd
from itertools import product

modes = ["reward", "value"]
steps = [0, 1, 2, 5, 10, 15, 20]
thresholds = [2, 1.5, 1]
inner_steps = [0, 1, 2, 5]

metric = "actor_critic/eval/planned_cumulative_reward"
meta_metric = "meta_planning_depth"
eval_step_key = "eval_step"
num_planning_steps_key = "actor_critic/eval/num_planning_steps"

def get_last_value(run):
    try:
        hist = run.history(keys=[metric], pandas=True)
        if not hist.empty:
            return hist[metric].dropna().values[-1]
    except Exception:
        pass
    return None

def get_last_eval_step(run):
    """Episode length = last eval_step for the run."""
    try:
        hist = run.history(keys=[eval_step_key], pandas=True)
        if not hist.empty:
            vals = hist[eval_step_key].dropna().values
            if len(vals) > 0:
                return float(vals[-1])
    except Exception:
        pass
    return None

def get_num_planning_steps(run):
    """Single scalar; prefer summary, fall back to last history value."""
    try:
        v = run.summary.get(num_planning_steps_key, None)
        if v is not None:
            return float(v)
    except Exception:
        pass
    try:
        hist = run.history(keys=[num_planning_steps_key], pandas=True)
        if not hist.empty:
            vals = hist[num_planning_steps_key].dropna().values
            if len(vals) > 0:
                return float(vals[-1])
    except Exception:
        pass
    return None

def get_meta_depth_counts(run):
    """Counts of meta_planning_depth occurrences per run for depths 1..4."""
    try:
        hist = run.history(keys=[meta_metric], pandas=True)
        if not hist.empty:
            vals = hist[meta_metric].dropna().astype(int).values
            return {d: int(np.sum(vals == d)) for d in [1, 2, 3, 4]}
    except Exception:
        pass
    return {d: 0 for d in [1, 2, 3, 4]}

# Filter runs that DO start with 'pruned'
runs = [r for r in runs if r.name.startswith("pruned")]

results = []
for mode, step, thres, inner in product(modes, steps, thresholds, inner_steps):
    matched_records = []

    for run in runs:
        cfg = run.config
        if run.state != "finished":
            continue
        try:
            # enforce planning_depth condition
            required_depth = 3 if (step == 0 or inner == 0) else 5

            if (
                str(cfg["evaluation"]["planning_mode"]) == str(mode) and
                int(cfg["evaluation"]["planning_steps"]) == int(step) and
                float(cfg["evaluation"]["entropy_threshold"]) == float(thres) and
                int(cfg["evaluation"]["inner_planning_steps"]) == int(inner) and
                int(cfg["evaluation"]["planning_depth"]) == required_depth
            ):
                rec = {}
                rec["reward"] = get_last_value(run)
                rec["ep_len"] = get_last_eval_step(run)
                rec["meta_counts"] = get_meta_depth_counts(run)
                rec["num_planning_steps"] = get_num_planning_steps(run)

                # require reward & episode length to include the run
                if rec["reward"] is not None and rec["ep_len"] is not None:
                    matched_records.append(rec)
        except KeyError:
            continue

    if len(matched_records) >= 3:
        print(f"Matched {len(matched_records)} runs for mode={mode}, steps={step}, thres={thres}, inner={inner}")
        recs = matched_records[:3]

        arr_vals = np.array([r["reward"] for r in recs], dtype=float)
        arr_lens = np.array([r["ep_len"] for r in recs], dtype=float)

        # meta-depth counts per run
        counts_per_run = {d: np.array([r["meta_counts"][d] for r in recs], dtype=float) for d in [1, 2, 3, 4]}
        counts_mean = {d: float(np.mean(counts_per_run[d])) for d in [1, 2, 3, 4]}
        counts_std  = {d: float(np.std(counts_per_run[d]))  for d in [1, 2, 3, 4]}

        # num_planning_steps per run (allow NaN)
        nps = np.array([r["num_planning_steps"] if r["num_planning_steps"] is not None else np.nan for r in recs], dtype=float)
        nps_mean = float(np.nanmean(nps)) if np.any(~np.isnan(nps)) else None
        nps_std  = float(np.nanstd(nps))  if np.any(~np.isnan(nps)) else None

        result = {
            "mode": mode,
            "planning_steps": step,
            "threshold": thres,
            "inner_steps": inner,
            "reward_mean": arr_vals.mean(),
            "reward_std": arr_vals.std(),
            "ep_length_mean": arr_lens.mean(),
            "ep_length_std": arr_lens.std(),
            "num_planning_steps_mean": nps_mean,
            "num_planning_steps_std": nps_std,
            **{f"meta_depth_{d}_mean": counts_mean[d] for d in [1, 2, 3, 4]},
            **{f"meta_depth_{d}_std": counts_std[d] for d in [1, 2, 3, 4]},
        }
    else:
        result = {
            "mode": mode,
            "planning_steps": step,
            "threshold": thres,
            "inner_steps": inner,
            "reward_mean": None,
            "reward_std": None,
            "ep_length_mean": None,
            "ep_length_std": None,
            "num_planning_steps_mean": None,
            "num_planning_steps_std": None,
            **{f"meta_depth_{d}_mean": None for d in [1, 2, 3, 4]},
            **{f"meta_depth_{d}_std": None for d in [1, 2, 3, 4]},
        }

    results.append(result)

# Export
df = pd.DataFrame(results)
print(df)
df.to_csv("pruned_sweep_results.csv", index=False)

Matched 3 runs for mode=reward, steps=0, thres=2, inner=0
Matched 3 runs for mode=reward, steps=1, thres=2, inner=0
Matched 3 runs for mode=reward, steps=1, thres=2, inner=1
Matched 3 runs for mode=reward, steps=1, thres=2, inner=2
Matched 3 runs for mode=reward, steps=1, thres=2, inner=5
Matched 3 runs for mode=reward, steps=1, thres=1.5, inner=0
Matched 3 runs for mode=reward, steps=1, thres=1.5, inner=1
Matched 3 runs for mode=reward, steps=1, thres=1.5, inner=2
Matched 3 runs for mode=reward, steps=1, thres=1.5, inner=5
Matched 3 runs for mode=reward, steps=1, thres=1, inner=0
Matched 3 runs for mode=reward, steps=1, thres=1, inner=1
Matched 3 runs for mode=reward, steps=1, thres=1, inner=2
Matched 3 runs for mode=reward, steps=1, thres=1, inner=5
Matched 3 runs for mode=reward, steps=2, thres=2, inner=0
Matched 3 runs for mode=reward, steps=2, thres=2, inner=1
Matched 3 runs for mode=reward, steps=2, thres=2, inner=2
Matched 3 runs for mode=reward, steps=2, thres=2, inner=5
Matche